In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from pathlib import Path
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, confusion_matrix
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

In [ ]:
from models import build_encoder, build_decoder, VAE, LATENT_DIM, WIDTH_FIRST, BETA_KL, DROPOUT
enc = build_encoder(7800)
dec = build_decoder(7800)
vae = VAE(enc, dec)

print("=== Hiperparametry architektury (models.py) ===")
print(f"  LATENT_DIM  = {LATENT_DIM}")
print(f"  WIDTH_FIRST = {WIDTH_FIRST}")
print(f"  BETA_KL     = {BETA_KL}")
print(f"  DROPOUT     = {DROPOUT}")

In [ ]:
ROOT = Path(r"E:\marcel.furs\Documents\Data_Run_Walk\Data_Run_Walk")

In [ ]:
#ROOT_Exp = "exp1a_train_M_detect_F"
ROOT_Exp = "exp1b_train_F_detect_M"

#ROOT_Exp = "exp_train_walk_detect_run"
#ROOT_Exp = "exp_train_run_detect_walk"

#ROOT_Exp = "exp_train_slow_detect_fast"
#ROOT_Exp = "exp_train_fast_detect_slow"

#EXP_DIR = ROOT / "EXPERIMENTS" / ROOT_Exp

DATA_DIR_SEX = ROOT / "CLEAN_ANGLES_DATASET_FULLBODY"
SEED = 42

In [ ]:
DATA_DIR_SEX = ROOT / "CLEAN_ANGLES_DATASET_FULLBODY"

X_train_M = np.load(DATA_DIR_SEX / "train_M_S1_fullbody.npy")
X_test_M  = np.load(DATA_DIR_SEX / "test_M_S2_fullbody.npy")
X_train_F = np.load(DATA_DIR_SEX / "train_F_S1_fullbody.npy")
X_test_F  = np.load(DATA_DIR_SEX / "test_F_S2_fullbody.npy")

subj_train_M = np.load(DATA_DIR_SEX / "subjects_train_M_fullbody.npy", allow_pickle=True)
subj_test_M  = np.load(DATA_DIR_SEX / "subjects_test_M_fullbody.npy",  allow_pickle=True)
subj_train_F = np.load(DATA_DIR_SEX / "subjects_train_F_fullbody.npy", allow_pickle=True)
subj_test_F  = np.load(DATA_DIR_SEX / "subjects_test_F_fullbody.npy",  allow_pickle=True)

# Filtr: M S2 tylko osoby obecne w M S1 (np. DA013 - brak w S1)
train_M_subjects = np.unique(subj_train_M)
keep = np.isin(subj_test_M, train_M_subjects)
X_test_M_filt    = X_test_M[keep]
subj_test_M_filt = subj_test_M[keep]

# Anomaly = wszystkie F (S1 + S2 sklejone)
X_anomaly_all  = np.concatenate([X_train_F, X_test_F], axis=0)
subj_anomaly   = np.concatenate([subj_train_F, subj_test_F])

# Per-subject split 80/20 w obrębie M S1 (replikacja paperowego seed=42)
unique_train_M = np.unique(subj_train_M)
n_val = max(1, int(len(unique_train_M) * 0.2))

np.random.seed(42)
val_M_subj   = np.random.choice(unique_train_M, size=n_val, replace=False)
train_M_subj = np.array([s for s in unique_train_M if s not in val_M_subj])

mask_tr = np.isin(subj_train_M, train_M_subj)
mask_va = np.isin(subj_train_M, val_M_subj)

X_train_paper  = X_train_M[mask_tr]
X_val_paper    = X_train_M[mask_va]
X_test_normal  = X_test_M_filt
X_test_anomaly = X_anomaly_all

print(f"train:        {X_train_paper.shape}, {len(train_M_subj)} osób")
print(f"val:          {X_val_paper.shape}, {len(val_M_subj)} osób")
print(f"test_normal:  {X_test_normal.shape}, {len(np.unique(subj_test_M_filt))} osób")
print(f"test_anomaly: {X_test_anomaly.shape}, {len(np.unique(subj_anomaly))} osób")

In [ ]:
# F → M: train na F, anomaly = M
DATA_DIR_SEX = ROOT / "CLEAN_ANGLES_DATASET_FULLBODY"

X_train_F = np.load(DATA_DIR_SEX / "train_F_S1_fullbody.npy")
X_test_F  = np.load(DATA_DIR_SEX / "test_F_S2_fullbody.npy")
X_train_M = np.load(DATA_DIR_SEX / "train_M_S1_fullbody.npy")
X_test_M  = np.load(DATA_DIR_SEX / "test_M_S2_fullbody.npy")

subj_train_F = np.load(DATA_DIR_SEX / "subjects_train_F_fullbody.npy", allow_pickle=True)
subj_test_F  = np.load(DATA_DIR_SEX / "subjects_test_F_fullbody.npy",  allow_pickle=True)
subj_train_M = np.load(DATA_DIR_SEX / "subjects_train_M_fullbody.npy", allow_pickle=True)
subj_test_M  = np.load(DATA_DIR_SEX / "subjects_test_M_fullbody.npy",  allow_pickle=True)

# Filtr: F S2 tylko osoby obecne w F S1 (analogicznie do DA013, dla F prawdopodobnie 0)
train_F_subjects = np.unique(subj_train_F)
keep = np.isin(subj_test_F, train_F_subjects)
X_test_F_filt    = X_test_F[keep]
subj_test_F_filt = subj_test_F[keep]
print(f"Wykluczono z F S2: {(~keep).sum()} cykli")

# Anomaly = wszystkie M (S1 + S2)
X_anomaly_all = np.concatenate([X_train_M, X_test_M], axis=0)
subj_anomaly  = np.concatenate([subj_train_M, subj_test_M])

# Per-subject split 80/20 w obrębie F S1 (seed=42, paperowy)
unique_train_F = np.unique(subj_train_F)
n_val = max(1, int(len(unique_train_F) * 0.2))

np.random.seed(42)
val_F_subj   = np.random.choice(unique_train_F, size=n_val, replace=False)
train_F_subj = np.array([s for s in unique_train_F if s not in val_F_subj])

mask_tr = np.isin(subj_train_F, train_F_subj)
mask_va = np.isin(subj_train_F, val_F_subj)

X_train_paper  = X_train_F[mask_tr]
X_val_paper    = X_train_F[mask_va]
X_test_normal  = X_test_F_filt
X_test_anomaly = X_anomaly_all

print(f"train:        {X_train_paper.shape}, {len(train_F_subj)} osób")
print(f"val:          {X_val_paper.shape}, {len(val_F_subj)} osób")
print(f"test_normal:  {X_test_normal.shape}, {len(np.unique(subj_test_F_filt))} osób")
print(f"test_anomaly: {X_test_anomaly.shape}, {len(np.unique(subj_anomaly))} osób")

In [ ]:
# Source: EXPERIMENTS (paperowe pre-saved splits)
ROOT_Exp = "exp1b_train_F_detect_M"   
EXP_DIR = ROOT / "EXPERIMENTS" / ROOT_Exp

X_train_paper  = np.load(EXP_DIR / "X_train.npy")
X_val_paper    = np.load(EXP_DIR / "X_val.npy")
X_test_normal  = np.load(EXP_DIR / "X_test_normal.npy")
X_test_anomaly = np.load(EXP_DIR / "X_test_anomaly.npy")

subj_train = np.load(EXP_DIR / "subjects_train.npy", allow_pickle=True)
subj_val   = np.load(EXP_DIR / "subjects_val.npy",   allow_pickle=True)
subj_test_normal  = np.load(EXP_DIR / "subjects_test_normal.npy",  allow_pickle=True)
subj_test_anomaly = np.load(EXP_DIR / "subjects_test_anomaly.npy", allow_pickle=True)

# Rekonstrukcja S1 normal (do random per-subject splitu)
X_S1_normal     = np.concatenate([X_train_paper, X_val_paper])
subj_S1_normal  = np.concatenate([subj_train, subj_val])

print(f"S1 normal pool:  {X_S1_normal.shape}, {len(np.unique(subj_S1_normal))} osób")
print(f"S2 normal:       {X_test_normal.shape}, {len(np.unique(subj_test_normal))} osób")
print(f"anomaly all:     {X_test_anomaly.shape}, {len(np.unique(subj_test_anomaly))} osób")

In [ ]:
# Source: EXPERIMENTS (speed)
ROOT_Exp =  "exp_train_run_detect_walk"
EXP_DIR = ROOT / "EXPERIMENTS" / ROOT_Exp

X_train_paper  = np.load(EXP_DIR / "X_train.npy")
X_val_paper    = np.load(EXP_DIR / "X_val.npy")
X_test_normal  = np.load(EXP_DIR / "X_test_normal.npy")
X_test_anomaly = np.load(EXP_DIR / "X_test_anomaly.npy")

subj_train = np.load(EXP_DIR / "subjects_train.npy", allow_pickle=True)
subj_val   = np.load(EXP_DIR / "subjects_val.npy",   allow_pickle=True)
subj_test_normal  = np.load(EXP_DIR / "subjects_test_normal.npy",  allow_pickle=True)
subj_test_anomaly = np.load(EXP_DIR / "subjects_test_anomaly.npy", allow_pickle=True)

# Rekonstrukcja S1 normal (do random per-subject splitu)
X_S1_normal    = np.concatenate([X_train_paper, X_val_paper])
subj_S1_normal = np.concatenate([subj_train, subj_val])

print(f"S1 normal pool:  {X_S1_normal.shape}, {len(np.unique(subj_S1_normal))} osób")
print(f"S2 normal:       {X_test_normal.shape}, {len(np.unique(subj_test_normal))} osób")
print(f"anomaly all:     {X_test_anomaly.shape}, {len(np.unique(subj_test_anomaly))} osób")

In [ ]:
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
BETA_KL = 0.01 
EPOCHS = 1000
PATIENCE = 30

In [ ]:
def compute_mse(model, X):
    mu, logvar, z = model.encoder.predict(X, verbose=0)
    X_recon = model.decoder.predict(z, verbose=0)
    return np.mean((X - X_recon)**2, axis=1)

In [ ]:
def evaluate(X_train_raw, X_val_raw, X_test_normal_raw, X_test_anomaly_raw, seed):
    tf.keras.utils.set_random_seed(seed)

    # Flatten (N, 100, 26, 3) -> (N, 7800)
    X_train_flat        = X_train_raw.reshape(X_train_raw.shape[0], -1)
    X_val_flat          = X_val_raw.reshape(X_val_raw.shape[0], -1)
    X_test_normal_flat  = X_test_normal_raw.reshape(X_test_normal_raw.shape[0], -1)
    X_test_anomaly_flat = X_test_anomaly_raw.reshape(X_test_anomaly_raw.shape[0], -1)

    # Z-score: fit tylko na train
    scaler = StandardScaler()
    X_train_norm        = scaler.fit_transform(X_train_flat)
    X_val_norm          = scaler.transform(X_val_flat)
    X_test_normal_norm  = scaler.transform(X_test_normal_flat)
    X_test_anomaly_norm = scaler.transform(X_test_anomaly_flat)

    input_dim = X_train_norm.shape[1]

    # Build VAE
    encoder = build_encoder(input_dim)
    decoder = build_decoder(input_dim)
    vae = VAE(encoder, decoder, beta=BETA_KL)
    vae.compile(optimizer=keras.optimizers.Adam(LEARNING_RATE, clipnorm=1.0))

    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=0,
    )

    vae.fit(
        X_train_norm,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        shuffle=True,
        validation_data=(X_val_norm, None),
        callbacks=[early_stop],
        verbose=0,
    )

    # Reconstruction errors na test
    mse_test_normal  = compute_mse(vae, X_test_normal_norm)
    mse_test_anomaly = compute_mse(vae, X_test_anomaly_norm)

    # AUC + AP
    y_true  = np.concatenate([np.zeros(len(mse_test_normal)), np.ones(len(mse_test_anomaly))])
    y_score = np.concatenate([mse_test_normal, mse_test_anomaly])

    auc = roc_auc_score(y_true, y_score)
    ap  = average_precision_score(y_true, y_score)

    return {
        "auc": auc,
        "ap": ap,
        "n_train": len(X_train_norm),
        "n_val": len(X_val_norm),
        "n_test_normal": len(X_test_normal_norm),
        "n_test_anomaly": len(X_test_anomaly_norm),
    }

In [ ]:
import time
t0 = time.time()
result_paper = evaluate(X_train_paper, X_val_paper, X_test_normal, X_test_anomaly, seed=42)
print(f"czas: {time.time()-t0:.1f}s")
print(result_paper)

In [ ]:
def make_split(X_normal_S1, subj_normal_S1, seed):
    """Per-subject 80/20 split, zależny od seed."""
    unique_subj = np.unique(subj_normal_S1)
    n_val = max(1, int(len(unique_subj) * 0.2))

    rng = np.random.RandomState(seed)
    val_subj = rng.choice(unique_subj, size=n_val, replace=False)
    train_subj = np.array([s for s in unique_subj if s not in val_subj])

    mask_tr = np.isin(subj_normal_S1, train_subj)
    mask_va = np.isin(subj_normal_S1, val_subj)

    return X_normal_S1[mask_tr], X_normal_S1[mask_va]

In [ ]:
import time
import json

N_SPLITS = 30
seeds = list(range(N_SPLITS))

results = []
t0 = time.time()
for s in seeds:
    keras.backend.clear_session()
    #X_tr, X_va = make_split(X_train_M, subj_train_M, seed=s)
    #r = evaluate(X_tr, X_va, X_test_M_filt, X_anomaly_all, seed=s)
    X_tr, X_va = make_split(X_S1_normal, subj_S1_normal, seed=s)
    r = evaluate(X_tr, X_va, X_test_normal, X_test_anomaly, seed=s)
    results.append(r)
    print(f"seed {s:2d}: AUC={r['auc']:.3f}, AP={r['ap']:.3f}, "
          f"n_train={r['n_train']}, n_val={r['n_val']}")

aucs = [r["auc"] for r in results]
aps  = [r["ap"]  for r in results]
print(f"\nczas total: {time.time()-t0:.1f}s")
print(f"AUC: {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
print(f"AP:  {np.mean(aps):.3f} ± {np.std(aps):.3f}")

In [ ]:
def save_result(experiment, direction, config, model, aucs, aps,
                all_results_path="all_results.json"):

    entry = {
        "experiment": experiment,    # "sex" / "speed" / "walk_run"
        "direction": direction,      # "M_to_F", "F_to_M", "slow_to_fast", "walk_to_run", itd.
        "config": config,            # "A" / "B"
        "model": model,              # "dense_VAE" / "PCA" / "OCSVM" / "IF" / "LSTM_VAE"
        "n_splits": len(aucs),
        "aucs": list(map(float, aucs)),
        "aps":  list(map(float, aps)),
        "auc_mean": float(np.mean(aucs)),
        "auc_std":  float(np.std(aucs)),
        "ap_mean":  float(np.mean(aps)),
        "ap_std":   float(np.std(aps)),
    }
    
    if Path(all_results_path).exists():
        with open(all_results_path) as f:
            all_results = json.load(f)
    else:
        all_results = []
    
    # nadpisz jeśli ten sam eksperyment już był (np. re-run)
    key = (experiment, direction, config, model)
    all_results = [r for r in all_results
                   if (r["experiment"], r["direction"], r["config"], r["model"]) != key]
    all_results.append(entry)
    
    with open(all_results_path, "w") as f:
        json.dump(all_results, f, indent=2)
    
    print(f"✓ zapisano: {experiment}/{direction}/{config}/{model}")

In [ ]:
save_result(experiment="sex", direction="Run_to_Walk", config="A", model="dense_VAE",
            aucs=aucs, aps=aps)

In [ ]:
GDY BEDZIE WSZYSTKO:

In [ ]:
import pandas as pd
with open("all_results.json") as f:
    all_results = json.load(f)

df = pd.DataFrame([{
    "experiment": r["experiment"],
    "direction": r["direction"],
    "config": r["config"],
    "model": r["model"],
    "AUC (mean ± std)": f"{r['auc_mean']:.3f} ± {r['auc_std']:.3f}",
    "AP (mean ± std)":  f"{r['ap_mean']:.3f} ± {r['ap_std']:.3f}",
}) for r in all_results])

# kolejność: experiment -> direction -> config -> model
df = df.sort_values(["experiment", "direction", "config", "model"])
print(df.to_string(index=False))
# do paperu LaTeX:
# print(df.to_latex(index=False))
# do Excela:
# df.to_csv("results_table.csv", index=False)